In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import math

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
OUTPUT_DIR = ROOT / 'outputs' / '06_5_model_comparison_A'
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
MODELS = ['IF', 'AE', 'LOF']
TOP_RATES = [0.01, 0.02, 0.05]
EXPECTED_VAL_N = 1430


In [2]:
def find_source(filename, stages=(), prefer_full=False):
    # Trong cuộc trò chuyện, file user tải lên nhiều lần có đuôi (2).
    # Ưu tiên bản FULL mới nhất đã gửi; trên máy user ưu tiên output/.._full.
    if prefer_full:
        stem, ext = Path(filename).stem, Path(filename).suffix
        uploaded_newest = ROOT / f'{stem}(2){ext}'
        if uploaded_newest.is_file():
            return uploaded_newest
    folders = []
    if prefer_full:
        folders.extend([
            ROOT / 'outputs' / '06_4_joint_behavior_A_full',
            ROOT / 'outputs' / '06_4_joint_behavior_A_full_v1',
        ])
    for stage in stages:
        folders.extend([ROOT / 'outputs' / stage, ROOT / stage])
    folders.extend([ROOT, ROOT / 'outputs'])

    for folder in folders:
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    # File đính kèm có thể có (1), (2) khi được tải nhiều lần
    for suffix in ['(2)', '(1)']:
        stem, ext = Path(filename).stem, Path(filename).suffix
        for folder in folders:
            candidate = folder / f'{stem}{suffix}{ext}'
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        f'Không tìm thấy {filename}. Đặt file cạnh notebook hoặc trong thư mục outputs tương ứng. '
        'Không tự thay bằng file kết quả khác.'
    )


def source_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def top_keys(df, model, n):
    # Stable sort: tie-break theo ticker và date, không thay đổi giữa các lần chạy.
    cols = [f'score_{model.lower()}', 'ticker', 'date']
    ordered = df.sort_values(
        cols, ascending=[False, True, True], kind='mergesort'
    ).head(n)
    return set(zip(ordered['ticker'], ordered['date']))


def basic_integrity(data, val):
    assert len(data) == EXPECTED_VAL_N, f'Validation select thiếu/thừa dòng: {len(data)}'
    assert len(val) == EXPECTED_VAL_N
    assert data[['ticker', 'date']].duplicated().sum() == 0
    assert data['ticker'].nunique() == 13
    assert data[FEATURES + [f'score_{m.lower()}' for m in MODELS]].notna().all().all()
    assert np.isfinite(data[FEATURES + [f'score_{m.lower()}' for m in MODELS]].to_numpy(float)).all()
    assert set(map(tuple, val[['ticker', 'date']].to_numpy())) == set(map(tuple, data[['ticker', 'date']].to_numpy()))
    assert data['date'].min() >= '2025-04-02'
    assert data['date'].max() <= '2025-09-10'


def load_data():
    files = {
        'dev': find_source('03_model_ready_development.csv', ['03_feature_engineering']),
        'if': find_source('06_1_baseline_val_scores.csv', ['06_1_if_validation_A']),
        'ae': find_source('06_2_ae_val_select_scores_seed42.csv', ['06_2_ae_validation_A']),
        'lof': find_source('06_3_lof_baseline_val_scores.csv', ['06_3_lof_validation_A']),
        'full_scores': find_source('06_4_w_floor_scores.csv', ['06_4_joint_behavior_A_full'], prefer_full=True),
        'full_sensitivity': find_source('06_4_w_floor_sensitivity.csv', ['06_4_joint_behavior_A_full'], prefer_full=True),
        'synthetic_pairs': find_source('06_4_synthetic_pairs.csv', ['06_4_joint_behavior_A_full'], prefer_full=True),
        'synthetic_summary': find_source('06_4_synthetic_summary.csv', ['06_4_joint_behavior_A_full'], prefer_full=True),
        'full_config': find_source('06_4_run_config.json', ['06_4_joint_behavior_A_full'], prefer_full=True),
        'if_stability': find_source('06_1_stability_summary.csv', ['06_1_if_validation_A']),
        'ae_stability': find_source('06_2_ae_seed_pairwise.csv', ['06_2_ae_validation_A']),
        'lof_sensitivity': find_source('06_3_lof_config_comparison.csv', ['06_3_lof_validation_A']),
    }
    dev = pd.read_csv(files['dev'])
    val = dev.loc[(dev['split'] == 'val') & (dev['val_role'] == 'select')].copy()
    assert len(dev.loc[dev['split'] == 'train']) == 10387
    assert len(val) == EXPECTED_VAL_N
    assert dev['split'].isin(['train', 'val']).all(), 'Input phát triển có Test!'
    assert val[['ticker', 'date']].duplicated().sum() == 0
    assert val[FEATURES].notna().all().all()

    ifdf = pd.read_csv(files['if'])[['ticker','date','score_if']]
    aedf = pd.read_csv(files['ae'])[['ticker','date','score_ae']]
    lofdf = pd.read_csv(files['lof'])[['ticker','date','score_lof']]
    for name, arr in [('IF', ifdf), ('AE', aedf), ('LOF', lofdf)]:
        assert len(arr) == EXPECTED_VAL_N and not arr[['ticker','date']].duplicated().any(), f'{name}: sai khóa'

    data = (val[['ticker','date'] + FEATURES]
            .merge(ifdf, on=['ticker','date'], how='inner', validate='one_to_one')
            .merge(aedf, on=['ticker','date'], how='inner', validate='one_to_one')
            .merge(lofdf, on=['ticker','date'], how='inner', validate='one_to_one')
            .sort_values(['date','ticker']).reset_index(drop=True))
    basic_integrity(data, val)

    full_sensitivity = pd.read_csv(files['full_sensitivity'])
    full_scores = pd.read_csv(files['full_scores'])
    synthetic_pairs = pd.read_csv(files['synthetic_pairs'])
    synthetic_summary = pd.read_csv(files['synthetic_summary'])
    with files['full_config'].open(encoding='utf-8') as f:
        config = json.load(f)
    if config.get('ae_refit_in_w_floor') is not True:
        raise AssertionError('Đang đọc Bước 06.4 QUICK (AE=false), phải dùng bản FULL AE=true')
    assert len(full_sensitivity) == 15
    assert len(full_scores) == 5 * 3 * EXPECTED_VAL_N
    assert set(full_scores['model']) == set(MODELS)
    assert full_scores.groupby(['case','model']).size().eq(EXPECTED_VAL_N).all()
    assert not full_scores.duplicated(['ticker','date','case','model']).any()
    assert config['input_sha256'] == source_hash(files['dev']), 'Nguồn đặc trưng Bước 03 không khớp hash Bước 06.4'
    assert len(synthetic_pairs) == 936 and len(synthetic_summary) == 18
    assert not synthetic_pairs.duplicated(['ticker','date','scenario']).any()

    # Đối chiếu thực tế ba baseline W60 với kết quả điểm gốc.
    max_diffs = {}
    for model in MODELS:
        base = full_scores[(full_scores['case'] == 'W60_floorx1') & (full_scores['model'] == model)]
        merged = data[['ticker','date', f'score_{model.lower()}']].merge(
            base[['ticker','date','score']], on=['ticker','date'],validate='one_to_one'
        )
        assert len(merged) == EXPECTED_VAL_N
        max_diff = float(np.max(np.abs(merged[f'score_{model.lower()}'] - merged['score'])))
        max_diffs[model] = max_diff
        assert max_diff < 5e-4, f'{model} baseline Bước 06.4 khác quá nhiều: {max_diff}'
    return data, full_sensitivity, full_scores, synthetic_pairs, synthetic_summary, files, max_diffs


In [3]:
def rank_and_agreement(data):
    for model in MODELS:
        data[f'rank_{model.lower()}'] = data[f'score_{model.lower()}'].rank(
            ascending=False, method='first'
        ).astype(int)
    pair_rows = []
    consensus_rows = []
    pair_rho = {
        f'{a}-{b}': float(spearmanr(data[f'score_{a.lower()}'], data[f'score_{b.lower()}']).statistic)
        for a,b in combinations(MODELS,2)
    }

    for rate in TOP_RATES:
        k = math.ceil(len(data) * rate)
        keysets = {model: top_keys(data,model,k) for model in MODELS}
        for a,b in combinations(MODELS,2):
            common = len(keysets[a] & keysets[b])
            pair_rows.append({
                'top_rate':rate,'top_n':k,'model_a':a,'model_b':b,
                'spearman_all_val':pair_rho[f'{a}-{b}'],
                'intersection_count':common,
                'overlap_pct':100*common/k,
                'jaccard_pct':100*common/(2*k-common),
            })
        union = set.union(*keysets.values())
        triple = set.intersection(*keysets.values())
        consensus_rows.append({
            'top_rate':rate,'top_n':k,'all_three_count':len(triple),
            'any_model_count':len(union),
            'if_only_count':len(keysets['IF']-keysets['AE']-keysets['LOF']),
            'ae_only_count':len(keysets['AE']-keysets['IF']-keysets['LOF']),
            'lof_only_count':len(keysets['LOF']-keysets['IF']-keysets['AE']),
            'at_least_two_count':sum(sum(key in s for s in keysets.values()) >= 2 for key in union)
        })
        for model in MODELS:
            selected = set(keysets[model])
            data[f'is_{model.lower()}_top{round(100*rate)}'] = list(zip(data.ticker,data.date))
            data[f'is_{model.lower()}_top{round(100*rate)}'] = data[f'is_{model.lower()}_top{round(100*rate)}'].map(lambda key:key in selected)
        data[f'n_models_top{round(100*rate)}'] = sum(data[f'is_{model.lower()}_top{round(100*rate)}'].astype(int) for model in MODELS)

    return data, pd.DataFrame(pair_rows), pd.DataFrame(consensus_rows)


def subgroups_and_cases(data):
    extended = data.copy()
    extended['month'] = extended['date'].str[:7]
    rows = []
    for rate in TOP_RATES:
        label = f'top{round(100*rate)}'
        for model in MODELS:
            col = f'is_{model.lower()}_{label}'
            for month, group in extended.groupby('month',sort=True):
                rows.append({'top_rate':rate,'model':model,'month':month,'val_rows':len(group),
                             'top_rows':int(group[col].sum()),'top_share_pct':100*group[col].sum()/extended[col].sum()})
    month = pd.DataFrame(rows)
    ticker_rows = []
    date_rows = []
    for rate in [0.01,0.05]:
        label = f'top{round(rate*100)}'
        for ticker, g in extended.groupby('ticker',sort=True):
            row = {'top_rate':rate,'ticker':ticker,'val_rows':len(g)}
            for model in MODELS:
                row[f'{model}_count'] = int(g[f'is_{model.lower()}_{label}'].sum())
            ticker_rows.append(row)
        for date,g in extended.groupby('date',sort=True):
            row = {'top_rate':rate, 'date':date,'val_rows':len(g)}
            for model in MODELS:
                row[f'{model}_count'] = int(g[f'is_{model.lower()}_{label}'].sum())
            date_rows.append(row)
    ticker_df = pd.DataFrame(ticker_rows)
    date_df = pd.DataFrame(date_rows)
    # Các phiên IF thuộc Top1/Top5 nhưng hai mô hình khác không đồng thuận.
    disagreements = extended.loc[
        ((extended['is_if_top1']) & (extended['n_models_top1'] == 1))
        | ((extended['is_if_top5']) & (extended['n_models_top5'] == 1))
        | (extended['n_models_top1'] == 3)
        | (extended['n_models_top5'] == 3)
        | ((~extended['is_if_top5']) & (extended['n_models_top5'] == 2))
    ].copy()
    disagreements['interpretation_group'] = np.select([
        disagreements['n_models_top1'].eq(3),
        disagreements['is_if_top1'] & disagreements['n_models_top1'].eq(1),
        disagreements['n_models_top5'].eq(3),
        disagreements['is_if_top5'] & disagreements['n_models_top5'].eq(1),
        (~disagreements['is_if_top5']) & disagreements['n_models_top5'].eq(2),
    ], ['all3_top1','IF_only_top1','all3_top5','IF_only_top5','AE_LOF_not_IF_top5'], default='other')
    return extended, month, ticker_df, date_df, disagreements


In [4]:
def check_synthetic(pairs, summary):
    out=[]
    for scenario, g in pairs.groupby('scenario',sort=False):
        assert len(g)==156, (scenario,len(g))
        assert g['is_effective'].all()
        for model in MODELS:
            diff=g[f'delta_score_{model}'].to_numpy(float)
            effective=g['target_z_increased_abs'].to_numpy(bool)
            n_eff=int(effective.sum())
            assert n_eff>0
            row={'scenario':scenario,'model':model,'n_cases':len(g),
                 'n_with_larger_target_abs_z':n_eff,
                 'score_increased_pct':float(np.mean(diff>0)*100),
                 'score_increased_among_amplified_pct':float(np.mean(diff[effective]>0)*100)}
            out.append(row)
    recheck=pd.DataFrame(out)
    verified=summary.merge(recheck, on=['scenario','model'], suffixes=('_saved','_recomputed'),validate='one_to_one')
    assert len(verified)==18
    for field in ['n_cases','n_with_larger_target_abs_z','score_increased_pct','score_increased_among_amplified_pct']:
        assert np.allclose(verified[f'{field}_saved'],verified[f'{field}_recomputed'],atol=1e-6)
    return recheck


def check_sensitivity(scores, sensitivity, data):
    rows=[]
    for (case,model),group in scores.groupby(['case','model']):
        assert len(group)==len(data)
        merged=data[['ticker','date',f'score_{model.lower()}']].merge(
            group[['ticker','date','score']],on=['ticker','date'],validate='one_to_one'
        )
        assert len(merged)==len(data)
        ref=merged[f'score_{model.lower()}'].to_numpy(float)
        alt=merged['score'].to_numpy(float)
        spearman=float(spearmanr(ref,alt).statistic)
        vals={'case':case,'model':model,'spearman':spearman}
        for rate in [0.01,0.05]:
            n=math.ceil(len(data)*rate)
            sr=set(np.argsort(-ref,kind='stable')[:n]); sa=set(np.argsort(-alt,kind='stable')[:n])
            vals[f'top{round(rate*100)}_overlap_pct']=100*len(sr&sa)/n
        rows.append(vals)
    recomputed=pd.DataFrame(rows)
    compare=sensitivity.merge(recomputed,on=['case','model'],suffixes=('_saved','_recomputed'),validate='one_to_one')
    assert len(compare)==15
    for name in ['spearman','top1_overlap_pct','top5_overlap_pct']:
        assert np.allclose(compare[f'{name}_saved'],compare[f'{name}_recomputed'],atol=1e-5,rtol=0), f'sai {name}'
    return recomputed


def load_prior_diagnostics(files):
    ifstab = pd.read_csv(files['if_stability'])
    ae = pd.read_csv(files['ae_stability'])
    lof = pd.read_csv(files['lof_sensitivity'])
    ifbaseline=ifstab.loc[(ifstab['n_estimators']==200)&(ifstab['max_samples']==256)]
    assert len(ifbaseline)==1
    assert len(ae)==10
    assert len(lof)==6
    return {
        'if_seed_spearman_median':float(ifbaseline.iloc[0]['rho_median']),
        'if_top5_overlap_seed_median':float(ifbaseline.iloc[0]['top5_overlap_median_pct']),
        'ae_seed_spearman_median':float(ae['spearman'].median()),
        'ae_top5_overlap_seed_median':float(ae['top5_overlap_pct'].median()),
        'lof_k20_vs_k100_top1_overlap':float(lof.loc[(lof['n_neighbors']==100)&(lof['minkowski_p']==2),'top1_overlap_pct'].iloc[0]),
    }


In [5]:
def md_table(df, columns=None, max_rows=None):
    cols = columns or list(df.columns)
    table = df.loc[:,cols]
    if max_rows is not None:
        table=table.head(max_rows)
    def fmt(x):
        if isinstance(x,(float,np.floating)):
            return f'{x:.2f}'
        return str(x).replace('|','\\|')
    lines=['| '+' | '.join(map(str,cols))+' |', '| '+' | '.join(['---']*len(cols))+' |']
    for row in table.itertuples(index=False,name=None):
        lines.append('| '+' | '.join(fmt(v) for v in row)+' |')
    return '\n'.join(lines)


def create_plots(agreement, month, tickers, outdir):
    fig, ax = plt.subplots(figsize=(8.5,5))
    for (a,b),group in agreement.groupby(['model_a','model_b'],sort=False):
        ax.plot(group['top_n'],group['overlap_pct'],marker='o',label=f'{a}–{b}')
    ax.set_xlabel('Số phiên có điểm cao (Top-k)')
    ax.set_ylabel('Tỷ lệ trùng danh sách (%)')
    ax.set_title('Bước 06.5 — Đồng thuận IF, AE và LOF theo Top-k')
    ax.set_xticks(sorted(agreement['top_n'].unique()))
    ax.legend()
    ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(outdir/'06_5_pairwise_topk_overlap.png',dpi=160)
    plt.close(fig)

    m=month.loc[month['top_rate']==0.05].pivot(index='month',columns='model',values='top_rows').fillna(0)
    m=m.reindex(columns=MODELS)
    fig, ax = plt.subplots(figsize=(9,5))
    x=np.arange(len(m.index)); width=.25
    for i,model in enumerate(MODELS):
        ax.bar(x+(i-1)*width,m[model].to_numpy(),width,label=model)
    ax.set_xticks(x, m.index, rotation=30)
    ax.set_ylabel('Số phiên thuộc Top 5%')
    ax.set_title('Bước 06.5 — Các phiên điểm cao theo tháng')
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir/'06_5_top5_by_month.png',dpi=160)
    plt.close(fig)

    t=tickers.loc[tickers['top_rate']==0.05].set_index('ticker')
    t=t.reindex(columns=[f'{m}_count' for m in MODELS])
    fig,ax=plt.subplots(figsize=(10,5))
    x=np.arange(len(t.index));width=.25
    for i,model in enumerate(MODELS):
        ax.bar(x+(i-1)*width,t[f'{model}_count'].to_numpy(),width,label=model)
    ax.set_xticks(x,t.index,rotation=25)
    ax.set_ylabel('Số phiên Top 5%')
    ax.set_title('Bước 06.5 — Số phiên điểm cao theo mã')
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir/'06_5_top5_by_ticker.png',dpi=160)
    plt.close(fig)


def build_report(data,agreement,consensus,month,ticker,date,disagreements,
                 sensitivity,synthetic,prior,max_diffs,files):
    rate5=month[month['top_rate']==0.05]
    apr = rate5[rate5['month']=='2025-04'].set_index('model')
    triple5=int(consensus.loc[consensus['top_rate']==0.05,'all_three_count'].iloc[0])
    triple1=int(consensus.loc[consensus['top_rate']==0.01,'all_three_count'].iloc[0])
    n= len(data)
    apr_val=int(data['date'].str.startswith('2025-04').sum())
    w_comp=sensitivity.pivot(index='case',columns='model',values='top5_overlap_pct').reset_index()
    syn=synthetic.pivot(index='scenario',columns='model',values='score_increased_among_amplified_pct').reset_index()
    if_only = disagreements.loc[disagreements['interpretation_group']=='IF_only_top1',
                                ['ticker','date','rank_if','rank_ae','rank_lof']+FEATURES]
    if_only=if_only.sort_values('rank_if').head(8)
    lines=[
        '# Bước 06.5 — So sánh tổng hợp IF (mô hình chính), AE và LOF',
        '',
        '## Phạm vi và kiểm tra dữ liệu',
        f'- Đọc điểm IF/AE/LOF trên đúng **{n:,}** phiên Validation chọn cấu hình từ Bước 06.1–06.3.',
        '- Mỗi mô hình sử dụng cùng 4 đặc trưng của phương án A, W=60 và mức sàn gốc.',
        '- Bước 06.4 đã xác minh bản `RUN_AE_REFITS=True`: 15 tổ hợp, 21.450 điểm.',
        f'- Sai khác tối đa điểm baseline giữa Bước 06.4 và bảng điểm gốc: {max_diffs}.',
        '- **Không dùng Validation hiệu chỉnh, Test hay proxy để chọn mô hình**. Không huấn luyện lại trong Bước 06.5.',
        '',
        '## Đồng thuận và khác biệt',
        md_table(agreement[['top_n','model_a','model_b','spearman_all_val','intersection_count','overlap_pct','jaccard_pct']]),
        '',
        'Đồng thuận ba mô hình và phiên chỉ do một mô hình ưu tiên:',
        md_table(consensus),
        '',
        f'- Có **{triple1}/15** phiên được cả ba cùng xếp Top 1%; **{triple5}/72** phiên được cả ba cùng xếp Top 5%.',
        '- Đồng thuận không phải ground truth; bất đồng không chứng minh một mô hình sai.',
        '',
        '### Các trường hợp IF ưu tiên riêng trong Top 1% (để rà, không tự gán nhãn)',
        md_table(if_only) if len(if_only) else 'Không có phiên IF-only trong Top 1%.',
        '',
        '## Sự tập trung theo thời gian và theo mã',
        f'- Tháng 04/2025 chiếm {apr_val}/{n} phiên Validation, xấp xỉ {100*apr_val/n:.1f}%.',
        f'- Trong Top 5%, số phiên tháng 04/2025: IF {int(apr.loc["IF","top_rows"])}/72; AE {int(apr.loc["AE","top_rows"])}/72; LOF {int(apr.loc["LOF","top_rows"])}/72.',
        '- Các mô hình có thể cùng phản ứng với giai đoạn thị trường biến động, nhưng chưa xác nhận nguyên nhân.',
        '- Tỷ lệ theo mã khác nhau không tự chứng minh thiên lệch. Cần diễn giải cùng nền lịch sử riêng mã.',
        '',
        '## Phản ứng trước sáu tình huống OHLCV đã thử',
        'Bảng: tỷ lệ tăng điểm trong các trường hợp thực sự tăng độ lớn của thành phần mục tiêu (không phải Recall).',
        md_table(syn),
        '- Các phép biến đổi chỉ được kiểm tra logic OHLC, chưa kiểm tra đầy đủ quy định biên độ/bước giá HOSE.',
        '- Mẫu thử 156 phiên, mỗi mã–tháng có 2 phiên gốc. Cường độ một số biến đổi khá lớn.',
        '',
        '## Độ nhạy W và mức sàn (Top 5% trùng với baseline của cùng thuật toán)',
        md_table(w_comp),
        '- Đây là mức **khác biệt so với W60/floor×1**, không phải phép đo mô hình nào đúng nhất.',
        '- W=20/60/120 dùng số phiên Train khác nhau, nên vừa thay đầu vào vừa thay lượng mẫu huấn luyện.',
        '- Bộ dữ liệu đầu ra không đủ để khẳng định W=60 tối ưu; cần đối chiếu có kiểm soát nếu dùng W để chọn cấu hình.',
        '',
        '## Bằng chứng từ từng thuật toán',
        f'- IF: Spearman trung vị giữa 5 seed ở 200 cây/256 mẫu = **{prior["if_seed_spearman_median"]:.4f}**; Top 5% trùng trung vị = **{prior["if_top5_overlap_seed_median"]:.1f}%**.',
        f'- AE: Spearman trung vị 5 seed = **{prior["ae_seed_spearman_median"]:.4f}**; Top 5% trùng trung vị = **{prior["ae_top5_overlap_seed_median"]:.1f}%**. 1.500 epoch vẫn chạm giới hạn; chưa chứng minh hội tụ.',
        f'- LOF: k=20 vs k=100 trùng Top 1% **{prior["lof_k20_vs_k100_top1_overlap"]:.1f}%**; LOF có độ nhạy cao với số hàng xóm.',
        '- Ba thuật toán có thang điểm khác nhau: so thứ hạng, Top-k, hành vi, không so điểm số tuyệt đối.',
        '',
        '## Quyết định phương pháp sau Bước 06.5',
        '**Đề xuất có điều kiện: giữ phương án A làm baseline để trình bày và hiệu chỉnh ngưỡng:**',
        '- IF **mô hình chính**: 200 cây, 256 mẫu/cây, seed42.',
        '- AE **đối chiếu**: 4→2→4, Adam lr=0.001, MSE, seed42, checkpoint epoch1495.',
        '- LOF **đối chiếu**: k=20, Euclid, novelty=True.',
        '- Cửa sổ W=60, mức sàn Train ban đầu là cấu hình baseline, **không gọi tối ưu**.',
        '- Không tự chọn IF tốt nhất theo tỷ lệ tăng điểm giả lập hoặc độ đồng thuận.',
        '',
        '### Hai cảnh báo trước Bước 07/08',
        '1. Nếu mục tiêu là **lựa chọn W và mức sàn tối ưu**, cần phép kiểm tra độc lập với chỉ số đồng thuận; các phép thử W hiện mới mô tả độ nhạy. Có thể giữ W=60 như phương án định trước và báo cáo độ nhạy.',
        '2. **Test dự kiến từng được chạy ở notebook cũ**: phải rà mức ảnh hưởng tới thiết kế trước khi tuyên bố phép đánh giá Test độc lập.',
        '',
        'Bước 07 chỉ hiệu chỉnh ba ngưỡng trên Validation hiệu chỉnh **sau khi người nghiên cứu chốt các lựa chọn**; không sử dụng tập này để sửa cấu hình.',
        '',
        '## Dấu vết đầu vào',
    ]
    for key,path in files.items():
        lines.append(f'- `{key}`: `{path}` (sha256: `{source_hash(path)}`)')
    return '\n\n'.join(lines) + '\n'


In [6]:
def main():
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    data,sensitivity,full_scores,pairs,summary,files,max_diffs = load_data()
    data,agreement,consensus=rank_and_agreement(data)
    data,month,ticker,date,disagreements=subgroups_and_cases(data)
    synthetic_checked=check_synthetic(pairs,summary)
    sensitivity_checked=check_sensitivity(full_scores,sensitivity,data)
    prior=load_prior_diagnostics(files)

    data.to_csv(OUTPUT_DIR/'06_5_aligned_val_scores.csv',index=False,encoding='utf-8-sig')
    agreement.to_csv(OUTPUT_DIR/'06_5_pairwise_agreement.csv',index=False,encoding='utf-8-sig')
    consensus.to_csv(OUTPUT_DIR/'06_5_three_model_consensus.csv',index=False,encoding='utf-8-sig')
    month.to_csv(OUTPUT_DIR/'06_5_top_by_month.csv',index=False,encoding='utf-8-sig')
    ticker.to_csv(OUTPUT_DIR/'06_5_top_by_ticker.csv',index=False,encoding='utf-8-sig')
    date.to_csv(OUTPUT_DIR/'06_5_top_by_date.csv',index=False,encoding='utf-8-sig')
    disagreements.to_csv(OUTPUT_DIR/'06_5_review_cases.csv',index=False,encoding='utf-8-sig')
    synthetic_checked.to_csv(OUTPUT_DIR/'06_5_synthetic_verified.csv',index=False,encoding='utf-8-sig')
    sensitivity_checked.to_csv(OUTPUT_DIR/'06_5_w_floor_verified.csv',index=False,encoding='utf-8-sig')
    create_plots(agreement,month,ticker,OUTPUT_DIR)

    report=build_report(data,agreement,consensus,month,ticker,date,disagreements,
                        sensitivity,synthetic_checked,prior,max_diffs,files)
    (OUTPUT_DIR/'06_5_research_report.md').write_text(report,encoding='utf-8')
    config={'step':'06_5_model_comparison_A', 'if_main_model':True,
            'val_select_n':len(data),'features':FEATURES,'top_rates':TOP_RATES,
            'no_test_no_calibrate_no_proxy':True, 'no_model_refitting':True,
            'w_floor_full_ae_verified':True,
            'baseline_max_abs_difference':max_diffs,
            'sources':{key:{'path':str(f),'sha256':source_hash(f)} for key,f in files.items()},
            'limitations':['No actual labels','W window affects Train size','Top-k overlap is not accuracy',
                           'Synthetic OHLCV not fully checked against HOSE trading rules','Test historical reuse unresolved']}
    (OUTPUT_DIR/'06_5_analysis_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')

    print('\n=== KIỂM TRA INPUT ===')
    print('VALIDATION CHỌN CẤU HÌNH:',len(data),'phiên,',data.ticker.nunique(),'mã')
    print('Bản AE FULL: 15 cấu hình; số điểm:',len(full_scores))
    print('Sai khác điểm baseline:',max_diffs)
    print('\n=== SO SÁNH BA MÔ HÌNH ===')
    print(agreement[['top_n','model_a','model_b','spearman_all_val','intersection_count','overlap_pct']].round(3).to_string(index=False))
    print('\n=== ĐỒNG THUẬN 3 MÔ HÌNH ===')
    print(consensus.to_string(index=False))
    print('\n=== TOP5% THÁNG 04/2025 ===')
    print(month[(month.top_rate==0.05)&(month.month=='2025-04')][['model','top_rows','top_share_pct']].round(2).to_string(index=False))
    print('\nĐã lưu báo cáo và toàn bộ kết quả tại:',OUTPUT_DIR.resolve())
    return {'paired':agreement,'consensus':consensus,'month':month,'sensitivity':sensitivity,'synthetic':synthetic_checked}


In [7]:
if __name__ == '__main__':
    _results = main()



=== KIỂM TRA INPUT ===
VALIDATION CHỌN CẤU HÌNH: 1430 phiên, 13 mã
Bản AE FULL: 15 cấu hình; số điểm: 21450
Sai khác điểm baseline: {'IF': 0.0, 'AE': 0.0, 'LOF': 0.0}

=== SO SÁNH BA MÔ HÌNH ===
 top_n model_a model_b  spearman_all_val  intersection_count  overlap_pct
    15      IF      AE             0.807                   5       33.333
    15      IF     LOF             0.632                   7       46.667
    15      AE     LOF             0.546                   9       60.000
    29      IF      AE             0.807                  14       48.276
    29      IF     LOF             0.632                  12       41.379
    29      AE     LOF             0.546                  17       58.621
    72      IF      AE             0.807                  49       68.056
    72      IF     LOF             0.632                  46       63.889
    72      AE     LOF             0.546                  39       54.167

=== ĐỒNG THUẬN 3 MÔ HÌNH ===
 top_rate  top_n  all_three_count 